# Fashion-MNIST: dense networks vs convolutions

Five Keras models on the same 10-class image problem, built to isolate what
actually moves accuracy: **architecture** (dense vs convolutional), **depth**,
and **optimiser** (SGD vs Adam).

Every model uses the same fixed seeds, the same normalised inputs and the same
train/test split, so the differences between them are attributable to the one
thing being changed.

Dataset: Fashion-MNIST, 60,000 train and 10,000 test, 28x28 greyscale.
Downloaded automatically by Keras, so there is nothing to fetch by hand.


## Setup and preprocessing

In [ ]:
# import necessary libs
import numpy as np
import tensorflow as tf
from keras.datasets import fashion_mnist
from keras.models import Sequential
from keras.utils import to_categorical
from keras.layers import Dense, Flatten, Dropout, Conv2D, MaxPooling2D
from keras.constraints import max_norm
from keras.optimizers import SGD

In [ ]:
from keras.datasets import fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# reshape dataset to have a single channel
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1))
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1))

In [ ]:
# normalize pixel values from 0 - 255 to 0.0 - 1.0
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

In [ ]:
# one-hot encode the labels
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

## Part 1: dense-only networks

Flattening a 28x28 image into a 784-vector discards every spatial relationship
between pixels. These two models establish how far that gets you.

In [ ]:
# baseline model
tf.random.set_seed(123)
np.random.seed(125)

model = Sequential()
model.add(Flatten(input_shape=(28, 28, 1)))
model.add(Dense(128, activation='relu'))
model.add(Dense(10, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='sgd', metrics=['accuracy'])
model.summary()

history = model.fit(x_train, y_train_cat, validation_data=(x_test, y_test_cat), epochs=10, batch_size=32, verbose=2)

scores = model.evaluate(x_test, y_test_cat, verbose=0)
print("Test accuracy:", scores[1])

In [ ]:
# improved model, add one more hidden layer and switched optimizer to "adam"
tf.random.set_seed(123)
np.random.seed(125)

model2 = Sequential()
model2.add(Flatten(input_shape=(28, 28, 1)))
model2.add(Dense(256, activation='relu'))
model2.add(Dense(128, activation='relu'))
model2.add(Dense(10, activation='softmax'))

model2.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model2.summary()

history2 = model2.fit(x_train, y_train_cat, validation_data=(x_test, y_test_cat), epochs=10, batch_size=32, verbose=2)

scores2 = model2.evaluate(x_test, y_test_cat, verbose=0)
print("Test accuracy:", scores2[1])

| Model | Params | Optimiser | Epochs | Test accuracy |
|---|---|---|---|---|
| Dense 128 | 101,770 | SGD | 10 | 0.8527 |
| Dense 256 to 128 | 235,146 | Adam | 10 | **0.8843** |

+3.2 points. Two things changed at once here (an extra layer *and* SGD to Adam),
so this comparison alone cannot attribute the gain. Part 3 separates them.

The second model's validation loss stops improving around epoch 5 (0.3629) while
training accuracy keeps climbing to 0.9186: it is overfitting from the midpoint
onward, and the final 0.8843 is being carried by the early epochs.

## Part 2: convolutional networks with SGD

Convolutions keep spatial structure: a filter sees a 3x3 neighbourhood, so edges
and textures survive into the representation.

In [ ]:
# baseline model
tf.random.set_seed(123)
np.random.seed(125)

model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape=(28, 28, 1), padding='same', activation='relu', kernel_constraint=max_norm(3)))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same', kernel_constraint=max_norm(3)))
model.add(MaxPooling2D())
model.add(Flatten())
model.add(Dense(512, activation='relu', kernel_constraint=max_norm(3)))
model.add(Dropout(0.2))
model.add(Dense(10, activation='softmax'))

lrate = 0.002
epochs = 5
decay = lrate / epochs
sgd = SGD(learning_rate=lrate, momentum=0.7, decay=decay, nesterov=False)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])
model.summary()

history = model.fit(x_train, y_train_cat, validation_data=(x_test, y_test_cat), epochs=epochs, batch_size=64, verbose=2)

scores = model.evaluate(x_test, y_test_cat, verbose=0)
print("Test accuracy:", scores[1])

In [ ]:
# fine-tuned model
tf.random.set_seed(123)
np.random.seed(125)

model2b = Sequential()
model2b.add(Conv2D(32, (3, 3), input_shape=(28, 28, 1), activation='relu', padding='same'))
model2b.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model2b.add(MaxPooling2D())
model2b.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model2b.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model2b.add(MaxPooling2D())
model2b.add(Flatten())
model2b.add(Dense(512, activation='relu'))
model2b.add(Dropout(0.2))
model2b.add(Dense(10, activation='softmax'))

epochs = 6
lrate = 0.001
decay = lrate / epochs
sgd = SGD(learning_rate=lrate, momentum=0.9, decay=decay, nesterov=False)
model2b.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])
model2b.summary()

history2b = model2b.fit(x_train, y_train_cat, validation_data=(x_test, y_test_cat), epochs=epochs, batch_size=64, verbose=2)

scores2b = model2b.evaluate(x_test, y_test_cat, verbose=0)
print("Test accuracy:", scores2b[1])


| Model | Params | Optimiser | Epochs | Test accuracy |
|---|---|---|---|---|
| 2 conv + dense 512, max-norm | 3,226,474 | SGD lr=0.002 m=0.7 | 5 | 0.8526 |
| 4 conv (2 blocks) + dense 512 | 1,676,266 | SGD lr=0.001 m=0.9 | 6 | **0.8763** |

The result that should stand out: the first CNN scores **0.8526**, which is
indistinguishable from the simple 128-unit dense network (0.8527), despite
having **32x more parameters**. Convolutions by themselves bought nothing here.

The reason is the optimiser, not the architecture. At `lr=0.002` with SGD and
5 epochs, this model is still climbing steeply when training stops (train
accuracy 0.8551 and rising, validation loss still falling). It is undertrained,
not incapable.

The deeper CNN confirms it: more layers but *fewer* parameters (1.68M vs 3.23M,
because the second max-pool shrinks the feature map before the dense layer) and
a higher momentum, giving 0.8763. Still climbing at epoch 6.

Note: `decay=` was passed to `SGD` and silently ignored by current Keras, which
warns about it. The learning-rate schedule these models were meant to have never
actually ran.

## Part 3: deeper CNN with Adam

Same convolutional family, three blocks instead of two, and Adam instead of SGD.

In [ ]:
# Deeper CNN: 3 convolutional blocks + 2 dense layers, trained with Adam instead of SGD
tf.random.set_seed(123)
np.random.seed(125)

model3 = Sequential()
model3.add(Conv2D(32, (3, 3), input_shape=(28, 28, 1), activation='relu', padding='same'))
model3.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model3.add(MaxPooling2D())
model3.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model3.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model3.add(MaxPooling2D())
model3.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model3.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model3.add(MaxPooling2D())
model3.add(Flatten())
model3.add(Dense(1024, activation='relu', kernel_constraint=max_norm(3)))
model3.add(Dense(512, activation='relu', kernel_constraint=max_norm(3)))
model3.add(Dropout(0.2))
model3.add(Dense(10, activation='softmax'))

epochs = 8
model3.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model3.summary()

history3 = model3.fit(x_train, y_train_cat, validation_data=(x_test, y_test_cat), epochs=epochs, batch_size=64, verbose=2)

scores3 = model3.evaluate(x_test, y_test_cat, verbose=0)
print("Test accuracy:", scores3[1])

| Model | Params | Optimiser | Epochs | Test accuracy |
|---|---|---|---|---|
| 6 conv (3 blocks) + dense 1024 to 512 | 1,997,034 | **Adam** | 8 | **0.9141** |

**0.9141 is the best result, and the optimiser is doing most of the work.**

Compare like for like: the Part 2 CNN with SGD reached 0.8763 after 6 epochs and
was still climbing. This one passes that score **after a single epoch** (0.8783
validation) and converges by epoch 3 (0.9087). Adam's per-parameter adaptive
learning rates remove the hand-tuning of `lr` and `momentum` that the SGD models
needed and never got right.

Validation loss bottoms out at epoch 3 (0.2826) and drifts up to 0.3069 by
epoch 8 while validation accuracy stays flat near 0.914. That is classic mild
overfitting. An early-stopping callback on validation loss would have returned
roughly the same accuracy in under half the training time.

## Summary

| # | Architecture | Optimiser | Test accuracy |
|---|---|---|---|
| 1 | Dense 128 | SGD | 0.8527 |
| 2 | Dense 256 to 128 | Adam | 0.8843 |
| 3 | 2 conv + dense | SGD | 0.8526 |
| 4 | 4 conv + dense | SGD | 0.8763 |
| 5 | **6 conv + dense 1024 to 512** | **Adam** | **0.9141** |

Ranking the three variables by how much they moved the result:

1. **Optimiser.** Adam vs SGD is worth roughly 4 points on the same
   convolutional family, and more importantly removes the need to hand-tune the
   learning rate.
2. **Depth.** Within SGD, 4 conv layers beat 2 by 2.4 points.
3. **Convolution vs dense.** Worth nothing on its own (0.8526 vs 0.8527) and
   everything in combination: the convolutional model only pulls ahead once it
   is trained properly.

Parameter count predicted nothing. The 3.23M-parameter model was the joint worst.
